# modification
1) unfreeze last 2 blocks other than FCL
2) fedprox instead of fed avg
additonally

1) unfreezing 6th layer
2) added class weights balancer in the loss
3) added cosine scheduler
4) the lr is set by the server side optimizer so it will change only after the rounds not within each epochs so modifiying it to local optimizer

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [8]:
import os
import copy
from collections import OrderedDict, Counter
import numpy as np
from collections import OrderedDict
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import Subset
from collections import defaultdict
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import ImageFolder
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

from sklearn.metrics import classification_report

In [9]:
class Net(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        # Load the base EfficientNet model with pre-trained weights
        self.base = efficientnet_b3(weights=EfficientNet_B3_Weights.IMAGENET1K_V1)

        # 1. Freeze the entire backbone initially
        for p in self.base.parameters():
            p.requires_grad = False

        # 2. PARTIAL UNFREEZE: Unfreeze the last two blocks (features.7 and features.8)
        # EfficientNet features are in blocks (0 to 8). We unfreeze the last few.
        # Unfreeze features.7 (Block 7)
        for p in self.base.features[6].parameters():
            p.requires_grad = True

        for p in self.base.features[7].parameters():
            p.requires_grad = True

        # Unfreeze features.8 (Block 8, which contains the final convolution layer)
        for p in self.base.features[8].parameters():
            p.requires_grad = True

        # 3. Replace and Unfreeze the final fully connected (FC) classifier
        num_ftrs = self.base.classifier[-1].in_features
        self.base.classifier[-1] = nn.Linear(num_ftrs, num_classes)

        for p in self.base.classifier.parameters():
            p.requires_grad = True

    def forward(self, x):
        return self.base(x)

In [10]:
img_tf = Compose([
    Resize((256, 256)),
    ToTensor(),
    Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

In [11]:
DATA_ROOT = "/kaggle/input/distributed-dataset/Splitted_data"
BATCH_SIZE = 32
N_FOLDS = 5
SEED = 42

def load_client_data(client_id):
    path = os.path.join(DATA_ROOT, f"Client_{client_id}")
    dataset = ImageFolder(path, transform=img_tf)

    n = len(dataset)
    t = int(0.8 * n)
    v = n - t

    train_ds, val_ds = random_split(dataset, [t, v], generator=torch.Generator().manual_seed(42))
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=32)
    return train_loader, val_loader

def get_client_dataset(client_id):
    path = os.path.join(DATA_ROOT, f"Client_{client_id}")
    dataset = ImageFolder(path, transform=img_tf)
    return dataset

In [12]:
def get_stratified_folds(dataset, n_splits=N_FOLDS, seed=SEED):
    # ImageFolder stores labels as dataset.targets on some torchvision versions, otherwise extract
    if hasattr(dataset, "targets"):
        labels = np.array(dataset.targets)
    else:
        labels = np.array([dataset[i][1] for i in range(len(dataset))])

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds = list(skf.split(np.zeros(len(labels)), labels))
    # folds is list of (train_idx, val_idx)
    return folds

In [13]:
def average_state_dicts(state_dicts):
    avg = copy.deepcopy(state_dicts[0])
    n = len(state_dicts)
    for k in avg.keys():
        # accumulate
        for i in range(1, n):
            avg[k] = avg[k] + state_dicts[i][k]
        avg[k] = avg[k] / n
    return avg

In [14]:
def local_train(model, loader, epochs, lr, device):
    model.to(device)
    model.train()

    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()

    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = crit(model(x), y)
            loss.backward()
            opt.step()

    return model.state_dict()  # return updated weights

In [15]:
from sklearn.metrics import classification_report

def local_eval(model, data_loader, device):
    model.eval()
    model.to(device)
    total_loss = 0
    total = 0
    all_preds = []
    all_labels = []
    all_probs = [] # <--- New: To store probabilities for ROC AUC

    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            # For classification report
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            # For ROC AUC: Convert logits to probabilities
            probs = torch.softmax(outputs, dim=1)
            all_probs.extend(probs.cpu().numpy()) # <--- New: Store probabilities

    avg_loss = total_loss / len(data_loader)

    # Generate a detailed classification report
    # We remove zero_division=0 to see warnings if classes are completely missed
    report = classification_report(all_labels, all_preds, output_dict=True, zero_division=0)

    # Return predictions, labels, and probabilities
    return avg_loss, report, np.array(all_labels), np.array(all_probs)

In [21]:
def local_train_fedprox(model, loader, epochs, lr, mu, device):
    model.to(device)
    model.train()

    # Deepcopy the global model weights w^t for the proximal term calculation
    w_global = OrderedDict(copy.deepcopy(model.state_dict()))

    # 1. Instantiate Optimizer
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    # 2. Instantiate Local Scheduler
    # T_max is set to the number of local epochs for decay across the local training session
    local_scheduler = CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-7)

    # 3. Use the provided class_weights in CrossEntropyLoss
    crit = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()

            # --- A. Standard Loss (Cross-Entropy) ---
            loss = crit(model(x), y)

            # --- B. Proximal Term (FedProx) ---
            proximal_term = 0.0

            # Iterate through all local model parameters (w)
            for name, param in model.named_parameters():
                if name in w_global:
                    # Calculate ||w - w^t||^2 for each layer
                    # Note: We must move w_global[name] to the current device
                    # before calculating the difference.
                    proximal_term += torch.sum((param - w_global[name].to(device)) ** 2)

            # Add the FedProx term: Loss + (mu/2) * ||w - w^t||^2
            loss += (mu / 2.0) * proximal_term

            # --- C. Backpropagation ---
            loss.backward()
            opt.step()

        # 4. Step the Local Scheduler after each epoch
        # This will adjust the LR for the next epoch's optimization steps.
        local_scheduler.step()
        
        # Optional: Check the decay
        # print(f"  Epoch {epoch+1}/{epochs}: Local LR={local_scheduler.get_last_lr()[0]:.8f}")

    return model.state_dict() # return updated weights

In [22]:
def fed_avg(models):
    avg = copy.deepcopy(models[0])
    for k in avg.keys():
        for i in range(1, len(models)):
            avg[k] += models[i][k]
        avg[k] = avg[k] / len(models)
    return avg

In [23]:
metrics_history = defaultdict(lambda: defaultdict(list))

In [ ]:
import copy
import numpy as np
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import roc_auc_score
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

# --- Configuration (Assuming these variables are set globally) ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLIENTS = 3
ROUNDS = 10
LOCAL_EPOCHS = 10
LR = 0.0001
MU = 0.01
NUM_CLASSES = 6
N_FOLDS = 5
SEED = 42
BATCH_SIZE = 32

# NOTE: The helper functions (Net, get_client_dataset, get_stratified_folds, local_train_fedprox, fed_avg, local_eval, compute_class_weights_from_indices) are assumed to be defined in preceding cells.

# --------------------------------------------------------
# Load datasets + create folds for each client (unchanged)
# --------------------------------------------------------
client_datasets = {}
client_folds = {}

for cid in range(1, NUM_CLIENTS + 1):
    # Assuming get_client_dataset(cid) is defined elsewhere
    ds = get_client_dataset(cid) 
    client_datasets[cid] = ds
    # Assuming get_stratified_folds(ds, ...) is defined elsewhere
    folds = get_stratified_folds(ds, n_splits=N_FOLDS, seed=SEED)
    client_folds[cid] = folds
    print(f"Client {cid}: {len(ds)} samples, {len(folds)} folds created.")

# --------------------------------------------------------
# Main results containers (MODIFIED)
# --------------------------------------------------------
# Stores final macro-averaged cross-fold results (Mean ± Std)
fold_metrics_per_client = {cid: [] for cid in range(1, NUM_CLIENTS + 1)}

# NEW CONTAINER: Stores metrics for EVERY CLIENT, EVERY ROUND, EVERY FOLD
# Structure: client_id -> fold_id -> round -> {'loss': v, 'accuracy': v, ...}
client_round_metrics_history = defaultdict(lambda: defaultdict(dict)) 

# Container for the overall aggregated ROC AUC (across all clients' data)
# Structure: fold_id -> round -> [roc_auc_score]
overall_roc_auc_history = defaultdict(lambda: defaultdict(list))

# --------------------------------------------------------
# 5-FOLD FEDERATED TRAINING (Evaluation Step Modified)
# --------------------------------------------------------
print("\n===== STARTING FULL 5-FOLD FEDERATED VALIDATION (Client-wise Metrics) =====")

for fold_id in range(N_FOLDS):
    print(f"\n\n===================== FOLD {fold_id+1}/{N_FOLDS} =====================")

    # Assuming Net() is defined elsewhere
    global_model = Net().to(DEVICE)
    
    # Prepare client-specific validation loaders for this fold
    client_val_loaders = {}
    for cid in range(1, NUM_CLIENTS + 1):
        ds = client_datasets[cid]
        _, val_idx = client_folds[cid][fold_id]
        client_val_loaders[cid] = DataLoader(Subset(ds, val_idx), batch_size=BATCH_SIZE, shuffle=False)
        
    # Run a full FL training for this fold
    for r in range(ROUNDS):
        print(f"\n----- Fold {fold_id+1}: Round {r+1}/{ROUNDS} -----")

        client_updates = []
        
        # --- 1. Local Training ---
        for cid in range(1, NUM_CLIENTS + 1):
            
            print(f" Client {cid}: Training on fold {fold_id+1}")

            ds = client_datasets[cid]
            train_idx, _ = client_folds[cid][fold_id]

            train_loader = DataLoader(Subset(ds, train_idx), batch_size=BATCH_SIZE, shuffle=True)
            
            # Compute weights for the **training** subset (Assuming compute_class_weights_from_indices is defined)
            # cw = compute_class_weights_from_indices(ds, train_idx)

            local_model = copy.deepcopy(global_model)

            # Local FedProx training (Assuming local_train_fedprox is defined)
            updated_state = local_train_fedprox(
                local_model, train_loader,
                LOCAL_EPOCHS, LR, MU, DEVICE
            )

            client_updates.append(updated_state)

        # --- 2. FedAvg Aggregation (Assuming fed_avg is defined) ---
        new_global = fed_avg(client_updates)
        global_model.load_state_dict(new_global)
        
        # --- 3. Round-wise Evaluation on all clients' validation sets (MODIFIED) ---
        
        round_losses = []
        round_reports = []
        all_round_labels = []
        all_round_probs = []

        for cid in range(1, NUM_CLIENTS + 1):
            val_loader = client_val_loaders[cid]
            
            # Use the UPDATED global model for evaluation (Assuming local_eval is defined)
            loss, report, labels, probs = local_eval(global_model, val_loader, DEVICE) 

            # Store Client-Specific Metrics (KEY MODIFICATION)
            client_round_metrics_history[cid][fold_id][r] = {
                'loss': loss,
                'accuracy': report["accuracy"],
                'macro_precision': report["macro avg"]["precision"],
                'macro_recall': report["macro avg"]["recall"],
                'macro_f1': report["macro avg"]["f1-score"],
            }
            
            # Store data for overall ROC AUC calculation
            round_losses.append(loss)
            round_reports.append(report) # Used to check if we can still calculate client-avg metrics
            all_round_labels.extend(labels)
            all_round_probs.extend(probs)

        # --- 4. Overall Aggregation for Console Output and Overall ROC AUC ---
        avg_loss = np.mean(round_losses)
        avg_acc = np.mean([r["accuracy"] for r in round_reports])
        avg_macro_f1 = np.mean([r["macro avg"]["f1-score"] for r in round_reports])

        try:
            # Multi-class ROC AUC uses 'ovr' strategy, averaged across all clients' data
            avg_roc_auc = roc_auc_score(all_round_labels, all_round_probs, multi_class='ovr', average='macro')
        except ValueError as e:
            avg_roc_auc = 0.5 
            print(f"    Warning: Could not calculate ROC AUC for round {r+1}: {e}")
        
        # Store overall AUC for the separate plot
        overall_roc_auc_history[fold_id][r].append(avg_roc_auc)
        
        print(f"    Round {r+1} Aggregated Validation Results: Loss={avg_loss:.4f}, Accuracy={avg_acc:.4f}, Macro F1={avg_macro_f1:.4f}, Macro AUC={avg_roc_auc:.4f}")

    # --------------------------
    # AFTER ALL ROUNDS → Final Evaluation (Unchanged from original)
    # --------------------------
    print(f"\n========== Evaluating Fold {fold_id+1} Final Model ==========")
    
    # ... (Final evaluation logic for fold_metrics_per_client remains the same) ...
    for cid in range(1, NUM_CLIENTS + 1):
        val_loader = client_val_loaders[cid]
        ds = client_datasets[cid]
        _, val_idx = client_folds[cid][fold_id]
        
        # Re-evaluate with the final model weights
        loss, report, _, _ = local_eval(global_model, val_loader, DEVICE)

        fold_summary = {
            "loss": loss, "accuracy": report["accuracy"],
            "macro_precision": report["macro avg"]["precision"],
            "macro_recall": report["macro avg"]["recall"],
            "macro_f1": report["macro avg"]["f1-score"]
        }
        fold_metrics_per_client[cid].append(fold_summary)
        print(f"\nClient {cid} Final Fold {fold_id+1} Results: Acc={fold_summary['accuracy']:.4f}, F1={fold_summary['macro_f1']:.4f}")

# --------------------------------------------------------
# FINAL MACRO AVERAGE ACROSS ALL 5 FOLDS (PER CLIENT) - Unchanged
# --------------------------------------------------------
# ... (Final printout section remains the same) ...
print("\n\n=================== FINAL CROSS-FOLD RESULTS ===================")
for cid in range(1, NUM_CLIENTS + 1):
    records = fold_metrics_per_client[cid]
    accs = [r["accuracy"] for r in records]
    mf1 = [r["macro_f1"] for r in records]
    mpr = [r["macro_precision"] for r in records]
    mre = [r["macro_recall"] for r in records]
    print(f"\n------ CLIENT {cid} (Macro Average Across {N_FOLDS} Folds) ------")
    print(f"Accuracy Mean ± Std           : {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"Macro F1 Mean ± Std           : {np.mean(mf1):.4f} ± {np.std(mf1):.4f}")
    print(f"Precision Mean ± Std          : {np.mean(mpr):.4f} ± {np.std(mpr):.4f}")
    print(f"Macro Recall Mean ± Std       : {np.mean(mre):.4f} ± {np.std(mre):.4f}")
print("\n=============== DONE ===============\n")

Client 1: 1420 samples, 5 folds created.
Client 2: 1395 samples, 5 folds created.
Client 3: 1387 samples, 5 folds created.

===== STARTING FULL 5-FOLD FEDERATED VALIDATION (Client-wise Metrics) =====


===================== FOLD 1/5 =====================

----- Fold 1: Round 1/10 -----
 Client 1: Training on fold 1
 Client 2: Training on fold 1
 Client 3: Training on fold 1
    Round 1 Aggregated Validation Results: Loss=0.7646, Accuracy=0.7396, Macro F1=0.4138, Macro AUC=0.9099

----- Fold 1: Round 2/10 -----
 Client 1: Training on fold 1
 Client 2: Training on fold 1
 Client 3: Training on fold 1
    Round 2 Aggregated Validation Results: Loss=0.5842, Accuracy=0.8026, Macro F1=0.6087, Macro AUC=0.9441

----- Fold 1: Round 3/10 -----
 Client 1: Training on fold 1
 Client 2: Training on fold 1
 Client 3: Training on fold 1
    Round 3 Aggregated Validation Results: Loss=0.5501, Accuracy=0.8299, Macro F1=0.6871, Macro AUC=0.9522

----- Fold 1: Round 4/10 -----
 Client 1: Training on fol

In [ ]:
df_list = []

# Loop through the client-wise history
for cid, fold_data in client_round_metrics_history.items():
    for fold_id, round_data in fold_data.items():
        for r, metrics in round_data.items():
            data = {
                'Round': r + 1,
                'Fold': fold_id + 1,
                'Client_ID': cid,
                'Loss': metrics['loss'],
                'Accuracy': metrics['accuracy'],
                'Macro Precision': metrics['macro_precision'],
                'Macro Recall': metrics['macro_recall'],
                'Macro F1': metrics['macro_f1'],
            }
            df_list.append(data)

df = pd.DataFrame(df_list)

# Calculate the mean and standard deviation across FOLDS for each (Client, Round) pair
# This implements your requested aggregation: Avg Round N in all Folds for that client
df_mean = df.groupby(['Client_ID', 'Round']).mean().reset_index()
df_std = df.groupby(['Client_ID', 'Round']).std().reset_index()

# --- 2. Visualization Setup ---
sns.set_style("whitegrid")
# Create a color palette with enough colors for the metrics
metrics_to_plot = ['Accuracy', 'Macro F1', 'Macro Precision', 'Macro Recall', 'Loss']
palette = sns.color_palette("colorblind", len(metrics_to_plot))
client_ids = sorted(df['Client_ID'].unique())
NUM_CLIENTS = len(client_ids)

# --- 3. Plot: Client-Specific Metrics in Subplots ---
# Use one column of subplots, one row per client
fig, axes = plt.subplots(NUM_CLIENTS, 1, figsize=(12, 5 * NUM_CLIENTS), sharex=True)
if NUM_CLIENTS == 1:
    axes = [axes] # Handle case of single client

for i, client_id in enumerate(client_ids):
    ax = axes[i]
    client_df_mean = df_mean[df_mean['Client_ID'] == client_id]
    client_df_std = df_std[df_std['Client_ID'] == client_id]
    
    ax.set_title(f'Client {client_id}: Validation Metrics Over FL Rounds (Mean ± Std Dev across {N_FOLDS} Folds)', fontsize=14)

    # Plot each metric for the current client
    for j, metric in enumerate(metrics_to_plot):
        color = palette[j]
        style = '--' if metric == 'Loss' else '-'
        
        ax.plot(client_df_mean['Round'], client_df_mean[metric], 
                label=metric, color=color, linestyle=style, linewidth=2)
        
        # Add shaded area (Mean +/- Std Dev across 5 Folds)
        ax.fill_between(client_df_mean['Round'], 
                        client_df_mean[metric] - client_df_std[metric], 
                        client_df_mean[metric] + client_df_std[metric], 
                        color=color, alpha=0.1)
    
    ax.set_ylabel('Metric Value', fontsize=12)
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, linestyle='--', alpha=0.6)

axes[-1].set_xlabel('Communication Round', fontsize=14)
plt.xticks(np.arange(1, ROUNDS + 1, 1))
plt.tight_layout()
plt.show()


# --- 4. Plot: Overall Aggregate Macro ROC AUC Curve (Unchanged Logic, uses new container) ---
roc_auc_list = []
for fold_id, round_data in overall_roc_auc_history.items():
    for r, auc_list in round_data.items():
        # auc_list contains one value: the overall macro ROC AUC for the round
        roc_auc_list.append({
            'Round': r + 1,
            'Fold': fold_id + 1,
            'Macro ROC AUC': auc_list[0]
        })

df_roc = pd.DataFrame(roc_auc_list)
df_roc_mean = df_roc.groupby('Round').mean().reset_index()
df_roc_std = df_roc.groupby('Round').std().reset_index()

plt.figure(figsize=(10, 5))
metric = 'Macro ROC AUC'
palette = sns.color_palette("colorblind", 1) 

plt.plot(df_roc_mean['Round'], df_roc_mean[metric], label=metric, color=palette[0], linewidth=2.5)
plt.fill_between(df_roc_mean['Round'], 
                 df_roc_mean[metric] - df_roc_std[metric], 
                 df_roc_mean[metric] + df_roc_std[metric], 
                 color=palette[0], alpha=0.2)

plt.axhline(0.5, color='gray', linestyle=':', label='Random Guess (0.5)')
plt.title('Aggregate Macro ROC AUC Over FL Rounds (Mean ± Std Dev across 5 Folds)', fontsize=16)
plt.xlabel('Communication Round', fontsize=14)
plt.ylabel('Macro ROC AUC Score', fontsize=14)
plt.legend(loc='lower right', fontsize=12)
plt.xticks(np.arange(1, ROUNDS + 1, 1))
plt.ylim(0.4, 1.0)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()